# Nieuwe methode proberen

Do your imports

In [9]:
from pathlib import Path
import geopandas as gpd
import folium 
from shapely.geometry import LineString, Polygon, box
import rasterio
import matplotlib.pyplot as plt
from shapely.geometry import shape
import geopandas as gpd
import pandas as pd
from pathlib import Path
import pickle
from ra2ce.network.network_config_data.enums.aggregate_wl_enum import AggregateWlEnum
from ra2ce.network.network_config_data.enums.source_enum import SourceEnum
from ra2ce.network.network_config_data.network_config_data import (
    HazardSection,
    NetworkConfigData,
    NetworkSection,
    OriginsDestinationsSection
)



In [10]:
# specify the name of the path to the project folder where you created the RA2CE folder setup

root_dir = Path(r'N:\Projects\11209000\11209175\B. Measurements and calculations\Data\full_run\V_K_analyse')
assert root_dir.exists()
static_path = root_dir.joinpath("static_maxwd")
hazard_path = static_path.joinpath("hazard")
hazard_reprojected_path = hazard_path.joinpath("reprojected")
network_path = static_path.joinpath("network")
output_path = static_path.joinpath("output_graph")


# Data pre-processing

In [11]:
# some preliminary functions

def get_all_files(directory: str) -> list[Path]:
    p = Path(directory)
    return [file for file in p.iterdir() if file.is_file()]

def read_pickle(file_path: str):
    with open(file_path, 'rb') as file:
        data = pickle.load(file)
    return data

def read_gpkg_to_gdf(file_path: str, layer: str = None) -> gpd.GeoDataFrame:
    # Read the geopackage file into a GeoDataFrame
    gdf = gpd.read_file(file_path, layer=layer)
    return gdf

In [12]:
hazard_files = get_all_files(hazard_path)
hazard_crs = "EPSG:4326" # for the hackathon case => "EPSG:4326" 

for hazard_file in hazard_files:
    print (hazard_file)

N:\Projects\11209000\11209175\B. Measurements and calculations\Data\full_run\V_K_analyse\static_maxwd\hazard\Merge_maxWD_ZH_200mm_Nat.tif
N:\Projects\11209000\11209175\B. Measurements and calculations\Data\full_run\V_K_analyse\static_maxwd\hazard\RMM_merge_max_wd_merge_clipNL.tif


# First check if there are values below 0 and if so process it

In [13]:
import rasterio
import numpy as np
from pathlib import Path

# Function to process each raster file
def process_raster(hazard_file, output_dir):
    with rasterio.open(hazard_file) as src:
        # Read the data
        data = src.read(1)
        
        # Check if there are any values below zero
        if not np.any(data < 0):
            print(f"There are no no data values in {hazard_file.name}")
            return  # Skip processing
        
        # Create a mask for values below zero
        mask = data < 0.01
        
        # Set values below zero to NoData
        data[mask] = src.nodata
        
        # Update the metadata to reflect changes
        meta = src.meta.copy()
        
        # Write the modified data to a new file in the output directory
        output_file = output_dir / f"{hazard_file.stem}_processed.tif"
        with rasterio.open(output_file, 'w', **meta) as dst:
            dst.write(data, 1)
        
        print(f"Processed {hazard_file} to {output_file}")



In [16]:
hazard_path_processed = hazard_path.joinpath("processed")
for hazard_file in hazard_files:
    process_raster(hazard_file, hazard_path_processed)

Processed N:\Projects\11209000\11209175\B. Measurements and calculations\Data\full_run\V_K_analyse\static_maxwd\hazard\Merge_maxWD_ZH_200mm_Nat.tif to N:\Projects\11209000\11209175\B. Measurements and calculations\Data\full_run\V_K_analyse\static_maxwd\hazard\processed\Merge_maxWD_ZH_200mm_Nat_processed.tif
Processed N:\Projects\11209000\11209175\B. Measurements and calculations\Data\full_run\V_K_analyse\static_maxwd\hazard\RMM_merge_max_wd_merge_clipNL.tif to N:\Projects\11209000\11209175\B. Measurements and calculations\Data\full_run\V_K_analyse\static_maxwd\hazard\processed\RMM_merge_max_wd_merge_clipNL_processed.tif


In [17]:
hazard_files_processed = get_all_files(hazard_path_processed)
hazard_crs = "EPSG:4326" 

for hazard_file in hazard_files_processed:
    print (hazard_file)

N:\Projects\11209000\11209175\B. Measurements and calculations\Data\full_run\V_K_analyse\static_maxwd\hazard\processed\RMM_merge_max_wd_merge_clipNL_processed.tif
N:\Projects\11209000\11209175\B. Measurements and calculations\Data\full_run\V_K_analyse\static_maxwd\hazard\processed\Merge_maxWD_ZH_200mm_Nat_processed.tif


In [18]:
from rasterio.warp import calculate_default_transform, reproject, Resampling

# Define target CRS
dst_crs = 'EPSG:4326'

for hazard_file in hazard_files_processed:
    print(f"Reprojecting: {hazard_file.name}")
    
    dst_file = hazard_reprojected_path / f"{hazard_file.stem}_wgs84{hazard_file.suffix}"

    with rasterio.open(hazard_file) as src:
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds)

        kwargs = src.meta.copy()
        kwargs.update({
            'crs': dst_crs,
            'transform': transform,
            'width': width,
            'height': height,
            'nodata': 0  # Set output nodata to 0
        })

        nodata = src.nodata

        with rasterio.open(dst_file, 'w', **kwargs) as dst:
            for i in range(1, src.count + 1):
                reprojected = np.empty((height, width), dtype=src.dtypes[i - 1])

                reproject(
                    source=rasterio.band(src, i),
                    destination=reprojected,
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=dst_crs,
                    resampling=Resampling.nearest)

                # Replace NaNs, nodata, and negative values with 0
                reprojected = np.where(
                    np.isnan(reprojected) | 
                    (reprojected < 0) | 
                    ((nodata is not None) & (reprojected == nodata)),
                    0,
                    reprojected
                )

                dst.write(reprojected, i)



Reprojecting: RMM_merge_max_wd_merge_clipNL_processed.tif
Reprojecting: Merge_maxWD_ZH_200mm_Nat_processed.tif
